# Swan · Étape 6 · Premier test sur le catalogue de Loredana

**Imane Moujahid**

Je relie les deux CSV v2 par code produit. Je vérifie ensuite la couverture France et j'exécute le classement et l'assemblage sur l'acné, la sécheresse et la sensibilité. Les sorties servent à vérifier le modèle en local.

Pour relancer, j’installe `requirements-notebooks.txt` et je place les trois entrées dans `raw_data/` à la racine du dépôt. Je récupère [les deux CSV v2 sur Slack](https://lewagon-alumni.slack.com/archives/D0CGCE62DPY/p1794353020215139) et l’extrait `Swan_CosIng_reference_2026-10-26.csv` utilisé par Imane. Je peux aussi indiquer un autre dossier avec `SWAN_DATA_DIR`. J’ouvre le notebook depuis la racine du dépôt ou depuis `notebooks/`, puis je lance toutes les cellules.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

workspace = Path.cwd()
if workspace.name == "notebooks":
    workspace = workspace.parent

data_dir = Path(os.environ.get("SWAN_DATA_DIR", workspace / "raw_data"))
catalogue = pd.read_csv(data_dir / "catalogue_v2.csv", dtype="string")
matrix = pd.read_csv(data_dir / "ingredient_matrix_v2.csv", dtype="string")

- J'utilise `Path` pour retrouver le dossier du projet depuis sa racine ou depuis `notebooks`. Je peux définir `SWAN_DATA_DIR` pour essayer un futur catalogue avec le même format.
- Je lis les champs en texte, y compris les codes-barres. Un code qui commence par zéro conserve son zéro. Je convertis les indicateurs au moment où j'en ai besoin.
- J'utilise `display` pour afficher les résultats sous forme de tableaux dans le notebook.

In [ ]:
required_fields = {
    "code", "role", "product_name", "france", "is_recommendable",
    "catalogue_review_complete", "has_parfum", "has_essential_oil",
    "ingredients_text", "inci_text", "has_acne_active", "is_gentle",
}
missing_fields = required_fields - set(catalogue.columns)
if missing_fields or "code" not in matrix:
    raise ValueError(f"Vérifier les colonnes d'entrée. Catalogue : {sorted(missing_fields)}.")

for name, table in (("catalogue", catalogue), ("matrice", matrix)):
    codes = table["code"].str.strip()
    if codes.fillna("").eq("").any() or codes.duplicated().any():
        raise ValueError(f"Codes manquants ou répétés dans le fichier {name}.")
    table["code"] = codes

if set(catalogue["code"]) != set(matrix["code"]):
    raise ValueError("Les deux fichiers doivent couvrir les mêmes codes produit.")

matrix_by_code = matrix.set_index("code").reindex(catalogue["code"])
print(f"{len(catalogue)} produits alignés, {matrix_by_code.shape[1]} colonnes d'ingrédients.")

- `required_fields` liste les colonnes que j'utilise. Si une future version change un nom, le notebook me signale le problème avant de classer.
- Je retire les espaces autour des codes et je refuse les codes vides ou répétés.
- Je compare les ensembles de codes, puis j'aligne la matrice dans l'ordre du catalogue avec `reindex`. Elle reste accessible par code pour retrouver des ingrédients lors d'une revue. Les règles de score ci-dessous utilisent les champs du catalogue et CosIng.

## Reprendre le moteur de l'étape 5

Je reprends les cellules de règles, de classement et d'assemblage pour que le notebook puisse tourner seul. `rank_by_role` filtre, ajoute les points, puis trie par score décroissant et code croissant. `choose_products` privilégie la couverture des rôles puis la somme des rangs. `assemble_routine` restitue les choix et les raisons.

In [ ]:
SCORING_RULES = [
    ("acne_prone", "cleanser", None, 0, "Signal acné par rôle à documenter"),
    ("acne_prone", "moisturizer", "has_acne_active", 1, "Signal du MVP, uniquement si revu dans le nouveau catalogue"),
    ("acne_prone", "sunscreen", None, 0, "Protection déclarée, aucun bonus acné provisoire"),
    ("dryness_barrier", "cleanser", None, 0, "Critère de douceur à définir avec Lina"),
    ("dryness_barrier", "moisturizer", "signal_hydratant", 1, "Fonction hydratante ou humectante détectée"),
    ("dryness_barrier", "sunscreen", "signal_hydratant", 1, "Bonus après validation du rôle et de la protection"),
    ("sensitivity_irritation", "cleanser", None, 0, "Appliquer les exclusions demandées"),
    ("sensitivity_irritation", "moisturizer", None, 0, "Appliquer les exclusions demandées"),
    ("sensitivity_irritation", "sunscreen", None, 0, "Appliquer les exclusions après validation du rôle"),
]

rules = pd.DataFrame(SCORING_RULES, columns=["concern", "role", "signal", "weight", "condition"])
display(rules)

In [ ]:
ROUTINE_ROLES = ("cleanser", "moisturizer", "sunscreen")
EXCLUSION_FIELDS = {"fragrance": "has_parfum", "essential_oils": "has_essential_oil"}


def as_flag(values):
    labels = values.astype("string").str.strip().str.lower()
    return labels.map({"true": True, "false": False}).astype("boolean")


def rank_by_role(products, concern, *, avoid=(), rules=rules):
    if concern not in rules["concern"].unique():
        raise ValueError("Préoccupation non reconnue.")
    if isinstance(avoid, str) or any(item not in EXCLUSION_FIELDS for item in avoid):
        raise ValueError("Exclusion non reconnue.")

    candidates = products.copy()
    candidates["code"] = candidates["code"].astype("string").str.strip()
    invalid_code = candidates["code"].fillna("").eq("")
    if invalid_code.any() or candidates.duplicated(["role", "code"]).any():
        raise ValueError("Le catalogue doit avoir un code par produit dans chaque rôle.")

    results = {}
    for role in ROUTINE_ROLES:
        eligible = candidates.loc[candidates["role"].eq(role)].copy()
        eligible = eligible.loc[as_flag(eligible["role_verified"]).eq(True).fillna(False)]
        if role == "sunscreen":
            eligible = eligible.loc[as_flag(eligible["sun_protection_verified"]).eq(True).fillna(False)]
        for item in avoid:
            field = EXCLUSION_FIELDS[item]
            if field not in eligible:
                eligible = eligible.iloc[0:0]
            else:
                eligible = eligible.loc[as_flag(eligible[field]).eq(False).fillna(False)]

        eligible["score"] = 0
        eligible["signaux_appliques"] = [[] for _ in range(len(eligible))]
        matching_rules = rules.loc[rules["concern"].eq(concern) & rules["role"].eq(role)]
        for rule in matching_rules.itertuples(index=False):
            if rule.weight and rule.signal in eligible:
                earns_points = as_flag(eligible[rule.signal]).eq(True).fillna(False)
                eligible.loc[earns_points, "score"] += rule.weight
                eligible.loc[earns_points, "signaux_appliques"] = (
                    eligible.loc[earns_points, "signaux_appliques"].map(lambda signals: signals + [rule.signal])
                )

        results[role] = eligible.sort_values(["score", "code"], ascending=[False, True])
    return results

In [ ]:
from itertools import product


def choose_products(ranked):
    options = []
    max_choices = len(ROUTINE_ROLES)
    for role in ROUTINE_ROLES:
        rows = ranked[role].head(max_choices).to_dict(orient="records")
        options.append(list(enumerate(rows, start=1)) + [(max_choices + 1, None)])

    best_key = None
    best_selection = None
    for selection in product(*options):
        codes = [row["code"] for _, row in selection if row is not None]
        if len(codes) != len(set(codes)):
            continue

        positions = tuple(position for position, _ in selection)
        rank_sum = sum(position for position, row in selection if row is not None)
        key = (-len(codes), rank_sum, positions)
        if best_key is None or key < best_key:
            best_key = key
            best_selection = selection

    return best_selection

In [ ]:
SIGNAL_LABELS = {
    "signal_hydratant": "fonction hydratante ou humectante détectée",
    "has_acne_active": "signal acné renseigné dans le catalogue v2",
}


def assemble_routine(ranked):
    selection = choose_products(ranked)
    steps = []
    for role, (position, row) in zip(ROUTINE_ROLES, selection):
        if row is None:
            reason = (
                "Aucun candidat admissible pour ce rôle."
                if ranked[role].empty
                else "Le code du candidat disponible est déjà utilisé dans la routine."
            )
            steps.append({"role": role, "code": None, "rang": None, "score": None, "raison": reason})
            continue

        labels = [SIGNAL_LABELS.get(signal, signal) for signal in row["signaux_appliques"]]
        reason = ", ".join(labels) if labels else "Produit admissible sans point de score pour ce profil."
        same_score = ranked[role]["score"].eq(row["score"]).sum()
        if position == 1 and same_score > 1:
            reason += f" Départagé par code produit parmi {same_score} candidats au même score."
        if position > 1:
            reason += f" Rang {position} dans ce rôle pour compléter la routine sans doublon."
        steps.append({
            "role": role,
            "code": row["code"],
            "rang": position,
            "score": int(row["score"]),
            "raison": reason,
        })

    filled = sum(step["code"] is not None for step in steps)
    status = {0: "no_result", len(ROUTINE_ROLES): "ok"}.get(filled, "partial")
    return {"status": status, "steps": steps}

## Préparer les entrées réelles du moteur

Je prends la France comme scénario de test à partir de la colonne `france`. Pour les sept solaires associés à la France, je documente le SPF déclaré par référence. Le marquage France du catalogue sert à limiter le test. La disponibilité en France devra être vérifiée avant diffusion.

In [ ]:
# Je saisis les valeurs lues sur les fiches et je garde leur lien pour les relire.
spf_review = pd.DataFrame([
    ("3468080009332", "50", "https://www.corinedefarme.fr/p/creme-protectrice-visage-et-corps-spf50/", "https://www.ezam.cz/pece-o-plet-a-pokozku/75210-opalovaci-krem-na-telo-a-oblicej-corine-de-farme-spf-50-50-ml-3468080009332.html", False),
    ("4743318143293", "30", "https://organicshop.co.uk/products/organic-shop-sunscreen-day-face-cream-30-spf-oily-skin-50ml", "https://www.pandashop.md/ru/product/sunscreen-day-face-cream-peach-spf30-50ml-00804499/", False),
    ("4005900458773", "50", "https://www.nivea.fr/produits/cr%C3%A8me-uv-visage-anti-%C3%A2ge-q10-spf-50-40060002197210040.html", "https://www.farmaline.be/fr/beaute/BE09802538/nivea-sun-uv-gezicht-anti-age-spf50.htm", False),
    ("42449010", pd.NA, pd.NA, pd.NA, False),
    ("3282770202120", "50+", "https://www.vidal.fr/parapharmacie/aderma-protect-spf-50-creme-sans-parfum-195613.html", pd.NA, True),
    ("4005900328946", "50+", "https://swiss-tablets.com/202394-nivea-sun-alpin-sun-cream-face-spf-50-50-ml/", pd.NA, False),
    ("3506770501008", "50", "https://www.bol.com/nl/nl/p/lovea-bio-anti-ageing-spf-50-50-ml-zonnebrand-creme/9200000005932746/", pd.NA, False),
], columns=["code", "spf_declare", "source_spf", "source_code", "arret_signale"])

reviewed = as_flag(catalogue["catalogue_review_complete"]).fillna(False)
recommended = as_flag(catalogue["is_recommendable"]).fillna(False)
france = as_flag(catalogue["france"]).fillna(False)

products = catalogue.loc[france].copy()
products["role_verified"] = (reviewed & recommended).loc[products.index]
products["sun_protection_verified"] = pd.Series(pd.NA, index=products.index, dtype="boolean")
if "sun_protection_verified" in catalogue:
    products["sun_protection_verified"] = as_flag(catalogue.loc[products.index, "sun_protection_verified"])

In [ ]:
solaires = products.loc[products["role"].eq("sunscreen") & products["role_verified"]]
spf_audit = solaires[["code", "product_name"]].merge(
    spf_review, on="code", how="left", validate="one_to_one"
)
verified_codes = spf_audit.loc[spf_audit["spf_declare"].notna(), "code"]
discontinued_codes = spf_audit.loc[spf_audit["arret_signale"].eq(True), "code"]

# Les données revues dans un nouvel export ont la priorité.
missing_spf = products["sun_protection_verified"].isna()
products.loc[missing_spf & products["code"].isin(verified_codes), "sun_protection_verified"] = True
products = products.loc[~products["code"].isin(discontinued_codes)].copy()
display(spf_audit[["code", "product_name", "spf_declare", "arret_signale"]])

In [ ]:
eligible_by_catalogue = catalogue.loc[reviewed & recommended]
display(pd.DataFrame({
    "catalogue_revu": eligible_by_catalogue.groupby("role").size(),
    "associe_a_la_france": catalogue.loc[reviewed & recommended & france].groupby("role").size(),
}).reindex(ROUTINE_ROLES).fillna(0).astype(int))

- `as_flag` reconnaît les libellés vrai et faux et conserve un état inconnu. Pour être retenu dans le test France et passer la revue du rôle, je demande un indicateur positif pour chacun des critères.
- Je relie la revue SPF au catalogue par code produit et je garde la décision du futur export lorsqu'elle est renseignée. Pour le Nivea SPF 30, je n'ai pas pu relier une fiche au code exact et je laisse le champ inconnu.
- J'écarte l'A-Derma du test de classement car Vidal signale un arrêt de commercialisation pour ce code. Je conserve son SPF dans le tableau de revue.
- Le tableau compte les décisions positives par rôle sur tout le catalogue, puis pour le scénario France.

## Calculer les signaux et tracer les exclusions

Je reprends le calcul CosIng de l'étape 5 pour la sécheresse. Pour le parfum et les huiles essentielles, je conserve les indicateurs source et je traite leurs valeurs négatives comme inconnues pour ce premier test. Le notebook de Loredana montre que ces valeurs correspondent à une recherche sans résultat dans le texte normalisé. Je conserve donc les absences non vérifiées comme inconnues.

In [ ]:
import re
import unicodedata

ALIASES = {
    "WATER": "AQUA",
    "EAU": "AQUA",
    "AQUA/WATER/EAU": "AQUA",
    "AQUA/WATER": "AQUA",
    "WATER/AQUA": "AQUA",
    "WATER (AQUA)": "AQUA",
    "AQUA (WATER)": "AQUA",
    "AQUA (WATER/EAU)": "AQUA",
    "WATER/AQUA/EAU": "AQUA",
    "AQUA (WATER, EAU)": "AQUA",
    "FRAGRANCE": "PARFUM",
    "FRAGRANCE (PARFUM)": "PARFUM",
    "PARFUM (FRAGRANCE)": "PARFUM",
}
FRAGRANCE_MARKERS = {"PARFUM", "PERFUME", "AROMA"}
SEPARATORS = re.compile(r"(?<!\d),|,(?!\d)|[;\n•]")


def inci_key(name):
    normalized = unicodedata.normalize("NFKC", name)
    return " ".join(normalized.upper().strip().strip("•·").split())


def ingredient_keys(text):
    if pd.isna(text):
        return []
    text = re.sub(r"^\s*(?:ingredients|ingrédients|inci)\s*:\s*", "", text, flags=re.I)
    parts = []
    depth = 0
    start = 0
    for index, char in enumerate(text):
        if char == "(":
            depth += 1
        elif char == ")":
            depth = max(0, depth - 1)
        elif depth == 0 and SEPARATORS.match(text, index):
            parts.append(text[start:index])
            start = index + 1
    parts.append(text[start:])

    keys = [inci_key(part) for part in parts]
    return [ALIASES.get(key, key) for key in keys if key]

In [ ]:
cosing = pd.read_csv(data_dir / "Swan_CosIng_reference_2026-10-26.csv", dtype="string")
reference = cosing.assign(key=cosing["inci_name"].dropna().map(inci_key))
unique_names = reference.loc[reference["key"].notna() & ~reference["key"].duplicated(keep=False)]
functions_by_name = {
    row.key: {part.strip().upper() for part in row.functions.split("|") if part.strip()}
    for row in unique_names[["key", "functions"]].dropna().itertuples(index=False)
}
FUNCTION_GROUPS = {
    "signal_hydratant": {"MOISTURISING", "HUMECTANT", "SKIN CONDITIONING - HUMECTANT"},
    "filtre_uv_repere": {"UV FILTER"},
}


def detect_evidence(keys):
    found = {
        signal: sorted({key for key in keys if functions_by_name.get(key, set()) & functions})
        for signal, functions in FUNCTION_GROUPS.items()
    }
    found["parfum_repere"] = sorted(set(keys) & FRAGRANCE_MARKERS)
    return found



evidence = products["ingredients_text"].map(ingredient_keys).map(detect_evidence)
products["signal_hydratant"] = evidence.map(lambda row: bool(row["signal_hydratant"]))
products["has_acne_active"] = as_flag(products["has_acne_active"])
signals = pd.DataFrame(index=catalogue.index)
for field in EXCLUSION_FIELDS.values():
    signals[field + "_source"] = as_flag(catalogue[field])
    signals[field] = signals[field + "_source"].where(signals[field + "_source"].eq(True))

# Détecter aussi le libellé explicite suivi de ponctuation ou d'une référence.
fragrance_pattern = r"\b(?:PARFUM|FRAGRANCE|PERFUME|AROMA)\b"
fragrance_in_raw = catalogue["ingredients_text"].str.contains(
    fragrance_pattern, case=False, regex=True, na=False
)
signals.loc[fragrance_in_raw, "has_parfum"] = True
signals["is_gentle_source"] = as_flag(catalogue["is_gentle"])
signals["is_gentle"] = signals["is_gentle_source"].mask(signals["has_parfum"].eq(True).fillna(False), False)
products = products.drop(columns=[*EXCLUSION_FIELDS.values(), "is_gentle"]).join(signals)

disagreements = fragrance_in_raw & signals["has_parfum_source"].eq(False).fillna(False)
display(catalogue.loc[disagreements & reviewed & recommended, ["code", "product_name", "france"]])

- `ingredient_keys` découpe le texte INCI, gère les parenthèses et normalise les noms. `detect_evidence` retrouve les noms dont le référentiel indique une fonction hydratante ou humectante.
- Je donne au moteur le signal acné du CSV pour tester la règle provisoire. Je garde la valeur source de `is_gentle` et je désactive le signal lorsqu'un parfum est repéré. Le classement sensibilité actuel n'attribue pas encore de bonus de douceur.
- Pour chaque exclusion, je sauvegarde l'indicateur du CSV avec le suffixe `_source`. `where` conserve les présences signalées et place les autres cas en inconnu.
- `str.contains` cherche des termes de parfum écrits dans le texte brut, même quand ils sont suivis d'un point ou d'une référence. Le tableau affiche les désaccords à relire. Une absence de correspondance dans cette recherche laisse l'état inconnu.

## Vérifier le cas Bioderma

Je teste la référence réelle à part, car le catalogue ne l'associe pas à la France. Je vérifie qu'elle passe la revue du rôle, qu'elle est écartée avec le filtre parfum et que son signal de douceur corrigé est faux.

In [ ]:
bioderma_code = "3401395376935"
sample = catalogue.loc[catalogue["code"].eq(bioderma_code), ["code", "role"]].join(signals)
if not sample.empty:
    sample["role_verified"] = (reviewed & recommended).loc[sample.index]
    sample["sun_protection_verified"] = pd.NA
    row = sample.iloc[0]
    assert row["has_parfum"] == True
    assert row["is_gentle"] == False
    concern = "sensitivity_irritation"
    assert bioderma_code in rank_by_role(sample, concern)["cleanser"]["code"].tolist()
    assert bioderma_code not in rank_by_role(sample, concern, avoid=("fragrance",))["cleanser"]["code"].tolist()
    print("Bioderma : parfum détecté, exclusion appliquée, signal de douceur désactivé.")

## Exécuter les scénarios

Je lance chaque préoccupation sans exclusion, avec chaque exclusion seule, puis avec les deux ensemble. Je compare les nombres de candidats et le résultat de l'assemblage. Pour les profils sans exclusion, je regarde les noms, les raisons et le nombre de candidats au même score.

In [ ]:
summary_rows = []
routine_rows = []
shuffled_products = products.sample(frac=1, random_state=42)
for concern in rules["concern"].unique():
    for avoid in ((), ("fragrance",), ("essential_oils",), ("fragrance", "essential_oils")):
        ranked = rank_by_role(products, concern, avoid=avoid)
        routine = assemble_routine(ranked)
        assert routine == assemble_routine(rank_by_role(shuffled_products, concern, avoid=avoid))
        selected = [step["code"] for step in routine["steps"] if step["code"] is not None]
        assert len(selected) == len(set(selected))
        assert ranked["sunscreen"]["sun_protection_verified"].eq(True).fillna(False).all()
        assert not ranked["sunscreen"]["code"].isin(discontinued_codes).any()
        for item in avoid:
            field = EXCLUSION_FIELDS[item]
            assert all(table[field].eq(False).fillna(False).all() for table in ranked.values())

        summary_rows.append({
            "préoccupation": concern,
            "exclusion": ", ".join(avoid) or "aucune",
            **{role: len(table) for role, table in ranked.items()},
            "routine": routine["status"],
        })
        if not avoid:
            for step in routine["steps"]:
                same_score = ranked[step["role"]]["score"].eq(step["score"]).sum()
                routine_rows.append({"préoccupation": concern, **step, "même score": same_score})

display(pd.DataFrame(summary_rows).set_index(["préoccupation", "exclusion"]))
routine_details = pd.DataFrame(routine_rows).merge(
    products[["code", "product_name"]], on="code", how="left", validate="many_to_one"
)
with pd.option_context("display.max_columns", None):
    display(routine_details.set_index(["préoccupation", "role"])[
        ["product_name", "rang", "score", "même score", "raison"]
    ])

- Les deux boucles font varier la préoccupation et l'exclusion en utilisant les mêmes produits.
- `sample` mélange le catalogue pour vérifier que l'ordre des lignes du CSV n'influence pas la routine. Les autres contrôles vérifient l'unicité des produits, le SPF des solaires, l'arrêt signalé et l'absence confirmée pour chaque exclusion demandée.
- Le premier tableau indique la couverture et le statut de la routine. Le deuxième relie chaque étape à son produit, son rang et ses points.

Pour relire le score, je regarde les égalités. Lorsque plusieurs produits ont les mêmes points, le code produit fixe l'ordre. Je veux examiner avec Lina les cas où la règle acné fait remonter un hydratant anti-âge et ceux où la sensibilité donne zéro point à tous les candidats. Pour la suite, je veux valider avec l’équipe les critères de douceur et les données nécessaires pour confirmer une absence.